In [ ]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter

from itertools import combinations
from scipy.spatial.distance import euclidean

sns.set_theme(style="ticks", context="talk")

plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

In [ ]:
datasets = {

    "Control": {
        "spots": "trackmate_spots.csv",
        "edges": "trackmate_edges.csv",
        "tracks": "trackmate_tracks.csv"
    },

    "2 wt% PEG": {
        "spots": "trackmate_spots.csv",
        "edges": "trackmate_edges.csv",
        "tracks": "trackmate_tracks.csv"
    },

    "5 wt% PEG": {
        "spots": "trackmate_spots.csv",
        "edges": "trackmate_edges.csv",
        "tracks": "trackmate_tracks.csv"
    },

    "10 wt% PEG": {
        "spots": "trackmate_spots.csv",
        "edges": "trackmate_edges.csv",
        "tracks": "trackmate_tracks.csv"
    },

}

In [ ]:
# =========================
# DATA LOADER
# =========================

def load_trackmate_csv(path):

    # Read first row as column names
    df = pd.read_csv(path, header=0)

    # Remove the three descriptive rows
    df = df.iloc[3:].reset_index(drop=True)

    # Remove completely empty rows
    df = df.dropna(how="all")

    # Convert numeric columns where possible
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            pass

    return df

all_data = {}

for condition, files in datasets.items():

    spots = load_trackmate_csv(files["spots"])
    edges = load_trackmate_csv(files["edges"])
    tracks = load_trackmate_csv(files["tracks"])

    all_data[condition] = {
        "spots": spots,
        "edges": edges,
        "tracks": tracks
    }

    print(condition)
    print(f"Spots : {len(spots)}")
    print(f"Edges : {len(edges)}")
    print(f"Tracks: {len(tracks)}")
    print()

for condition in all_data:

    print("\n")
    print(condition)

    print("\nSPOTS")
    print(all_data[condition]["spots"].columns.tolist())

    print("\nEDGES")
    print(all_data[condition]["edges"].columns.tolist())

In [ ]:
# =========================
# MERGE DETECTION
# =========================

def detect_merges(spots, edges):

    target_counts = (
        edges.groupby("SPOT_TARGET_ID")
        .size()
        .reset_index(name="n_incoming")
    )

    merge_targets = target_counts[
        target_counts["n_incoming"] > 1
    ]

    if len(merge_targets) == 0:
        return pd.DataFrame()

    merges = spots.merge(
        merge_targets,
        left_on="ID",
        right_on="SPOT_TARGET_ID",
        how="inner"
    )

    return merges

merge_events = {}

for condition in all_data:

    spots = all_data[condition]["spots"]
    edges = all_data[condition]["edges"]

    merges = detect_merges(spots, edges)

    merge_events[condition] = merges

    print(
        f"{condition}: "
        f"{len(merges)} merge events"
    )

In [ ]:
# =========================
# CUMULATIVE MERGE PLOT
# =========================

plt.figure(figsize=(7,5))

for condition, merges in merge_events.items():

    if len(merges) == 0:
        continue

    merges = merges.sort_values("POSITION_T")

    merges["cumulative_merges"] = np.arange(
        1,
        len(merges)+1
    )

    sns.lineplot(
        data=merges,
        x="POSITION_T",
        y="cumulative_merges",
        label=condition,
        linewidth=2.5
    )

plt.xlabel("Time (s)")
plt.ylabel("Cumulative fusion events")

plt.legend(frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# MERGE SUMMARY
# =========================

summary = []

for condition in all_data:

    merges = merge_events[condition]

    tracks = all_data[condition]["tracks"]

    total_merges = len(merges)

    n_tracks = len(tracks)

    duration = tracks["TRACK_STOP"].max()

    summary.append({

        "Condition": condition,

        "Fusion events":
            total_merges,

        "Tracks":
            n_tracks,

        "Duration (s)":
            duration,

        "Fusion events/min":
            total_merges /
            duration *
            60
    })

summary_df = pd.DataFrame(summary)

summary_df

In [ ]:
# =========================
# ENCOUNTER DETECTION
# =========================

def detect_encounters(spots, tolerance=1.0):
    
    """
    Detect droplet-droplet contacts from TrackMate spots.

    Parameters:
    -----------
    spots : dataframe
        TrackMate spots dataframe
    
    tolerance : float
        Additional distance tolerance (microns)

    Returns:
    --------
    dataframe containing encounter candidates
    """

    encounters = []

    # Only keep real tracked droplets
    spots = spots.dropna(
        subset=[
            "TRACK_ID",
            "POSITION_X",
            "POSITION_Y",
            "RADIUS",
            "FRAME"
        ]
    )

    # Ensure correct types
    spots["TRACK_ID"] = spots["TRACK_ID"].astype(int)

    # Analyse frame-by-frame
    for frame, frame_data in spots.groupby("FRAME"):

        # Need at least two droplets
        if len(frame_data) < 2:
            continue

        droplets = frame_data[
            [
                "TRACK_ID",
                "POSITION_X",
                "POSITION_Y",
                "RADIUS"
            ]
        ].values

        # All droplet pairs
        for a, b in combinations(droplets, 2):

            track_a, x_a, y_a, r_a = a
            track_b, x_b, y_b, r_b = b

            distance = np.sqrt(
                (x_a-x_b)**2 +
                (y_a-y_b)**2
            )

            contact_distance = (
                r_a +
                r_b +
                tolerance
            )

            if distance <= contact_distance:

                encounters.append({

                    "FRAME": frame,

                    "TRACK_A": int(track_a),
                    "TRACK_B": int(track_b),

                    "X_A": x_a,
                    "Y_A": y_a,

                    "X_B": x_b,
                    "Y_B": y_b,

                    "RADIUS_A": r_a,
                    "RADIUS_B": r_b,

                    "DISTANCE": distance,

                    "CONTACT_DISTANCE": contact_distance
                })


    return pd.DataFrame(encounters)

In [ ]:
# =========================
# ENCOUNTER DATA SUMMARY
# =========================

encounter_events = {}

for condition in all_data:

    print(condition)

    spots = all_data[condition]["spots"]

    encounters = detect_encounters(
        spots,
        tolerance=1.0
    )

    encounter_events[condition] = encounters

    print(
        f"Encounter detections: {len(encounters)}"
    )
    print()

encounter_summary = []

for condition, encounters in encounter_events.items():

    encounter_summary.append({

        "Condition": condition,
        "Encounter detections": len(encounters)

    })

encounter_summary = pd.DataFrame(
    encounter_summary
)

encounter_summary

In [ ]:
# ========================================
# GROUP ENCOUNTERS TO PREVENT OVERCOUNTING
# ========================================

def group_encounter_events(encounters, max_gap=1):

    if len(encounters) == 0:
        return pd.DataFrame()

    encounters = encounters.copy()

    # Remove self-interactions
    encounters = encounters[
        encounters["TRACK_A"] != encounters["TRACK_B"]
    ]

    # Convert track IDs to integers
    encounters["TRACK_A"] = (
        encounters["TRACK_A"]
        .astype(int)
    )

    encounters["TRACK_B"] = (
        encounters["TRACK_B"]
        .astype(int)
    )

    # Create a unique identifier for each droplet pair
    encounters["PAIR"] = encounters.apply(
        lambda row: tuple(
            sorted(
                [
                    row["TRACK_A"],
                    row["TRACK_B"]
                ]
            )
        ),
        axis=1
    )

    events = []

    # Process each droplet pair separately
    for pair, group in encounters.groupby("PAIR"):

        group = group.sort_values("FRAME")

        frames = group["FRAME"].values

        event_start = frames[0]
        previous = frames[0]

        for frame in frames[1:]:

            # If the gap is too large, end the current event
            if frame - previous > max_gap + 1:

                events.append({

                    "TRACK_A": pair[0],
                    "TRACK_B": pair[1],

                    "START_FRAME": event_start,
                    "END_FRAME": previous,

                    "DURATION_FRAMES":
                        previous - event_start + 1
                })

                event_start = frame

            previous = frame

        # Add final event
        events.append({

            "TRACK_A": pair[0],
            "TRACK_B": pair[1],

            "START_FRAME": event_start,
            "END_FRAME": previous,

            "DURATION_FRAMES":
                previous - event_start + 1
        })

    return pd.DataFrame(events)

encounter_events_unique = {}

for condition, encounters in encounter_events.items():

    grouped = group_encounter_events(
        encounters
    )

    encounter_events_unique[condition] = grouped

    print(condition)
    print(
        "Unique encounters:",
        len(grouped)
    )
    print()

In [ ]:
# ====================================
# ENCOUNTER NORMALISATION AND SUMMARY
# ====================================

collision_summary = []

for condition in all_data:

    spots = all_data[condition]["spots"]
    
    collisions = collision_events_unique[condition]


    # number of unique droplets tracked
    n_droplets = (
        spots["TRACK_ID"]
        .nunique()
    )


    # duration
    duration = (
        spots["POSITION_T"]
        .max()
        -
        spots["POSITION_T"]
        .min()
    )


    possible_pairs = (
        n_droplets *
        (n_droplets - 1)
        /
        2
    )


    collision_summary.append({

        "Condition": condition,

        "Droplets":
            n_droplets,

        "Duration (s)":
            duration,

        "Unique collisions":
            len(collisions),

        "Collision rate (events/s)":
            len(collisions)/duration,

        "Pair-normalized collision rate":
            len(collisions)
            /
            (possible_pairs*duration)

    })


collision_summary = pd.DataFrame(
    collision_summary
)


collision_summary

In [ ]:
fusion_probability = []

for condition in all_data:

    n_collisions = len(
        collision_events_unique[condition]
    )

    n_fusions = len(
        merge_events[condition]
    )


    fusion_probability.append({

        "Condition": condition,

        "Collision events":
            n_collisions,

        "Fusion events":
            n_fusions,

        "Fusion probability":
            n_fusions / n_collisions
            if n_collisions > 0
            else np.nan
    })


fusion_probability = pd.DataFrame(
    fusion_probability
)


fusion_probability

In [ ]:
tracks = all_data["Control"]["tracks"]

print(tracks.columns.tolist())

In [ ]:
tracks.head()

In [ ]:
track_merge_summary = []

for condition in all_data:

    tracks = all_data[condition]["tracks"]


    fusion_tracks = tracks[
        tracks["NUMBER_MERGES"] > 0
    ]


    track_merge_summary.append({

        "Condition": condition,

        "Tracks with merges":
            len(fusion_tracks),

        "Total merges":
            fusion_tracks["NUMBER_MERGES"]
            .sum()

    })


track_merge_summary = pd.DataFrame(
    track_merge_summary
)


track_merge_summary

In [ ]:
fusion_tracks = {}

for condition in all_data:

    tracks = all_data[condition]["tracks"]

    fusion_tracks[condition] = tracks[
        tracks["NUMBER_MERGES"] > 0
    ][
        [
            "TRACK_ID",
            "TRACK_START",
            "TRACK_STOP",
            "NUMBER_MERGES"
        ]
    ].copy()


    print(condition)
    print(
        fusion_tracks[condition].head()
    )
    print()

In [ ]:
for condition in all_data:

    spots = all_data[condition]["spots"]

    n_frames = (
        spots["FRAME"]
        .max()
        -
        spots["FRAME"]
        .min()
        +
        1
    )

    duration = (
        spots["POSITION_T"]
        .max()
        -
        spots["POSITION_T"]
        .min()
    )


    print(condition)

    print(
        "Frames:",
        n_frames
    )

    print(
        "Duration (s):",
        duration
    )

    print(
        "Frame interval (s):",
        duration/n_frames
    )

    print()

In [ ]:
fusion_tracks = {}

for condition in all_data:

    tracks = all_data[condition]["tracks"]

    fusion_tracks[condition] = tracks[
        tracks["NUMBER_MERGES"] > 0
    ][
        [
            "TRACK_ID",
            "TRACK_START",
            "TRACK_STOP",
            "NUMBER_MERGES"
        ]
    ].copy()

In [ ]:
def classify_collision_fusion(
    collision_events,
    fusion_tracks,
    frame_interval
):

    results = []


    for _, collision in collision_events.iterrows():

        track_a = collision["TRACK_A"]
        track_b = collision["TRACK_B"]


        # Convert frames to seconds
        t_start = (
            collision["START_FRAME"]
            *
            frame_interval
        )

        t_end = (
            collision["END_FRAME"]
            *
            frame_interval
        )


        # Find fusion tracks
        possible_fusions = fusion_tracks[
            fusion_tracks["TRACK_ID"]
            .isin(
                [
                    track_a,
                    track_b
                ]
            )
        ]


        fusion = False


        for _, track in possible_fusions.iterrows():

            merge_time = track["TRACK_STOP"]


            if (
                merge_time >= t_start
                and
                merge_time <= t_end + 10
            ):
                fusion = True



        results.append({

            "TRACK_A": track_a,
            "TRACK_B": track_b,

            "START_TIME": t_start,
            "END_TIME": t_end,

            "FUSION": fusion

        })


    return pd.DataFrame(results)

In [ ]:
collision_fusion_results = {}


frame_intervals = {

    "Control": 0.4393,
    "2 wt% PEG": 0.4359,
    "5 wt% PEG": 0.4394,
    "10 wt% PEG": 0.4590

}


for condition in all_data:

    results = classify_collision_fusion(

        collision_events_unique[condition],

        fusion_tracks[condition],

        frame_intervals[condition]

    )


    collision_fusion_results[condition] = results


    print(condition)

    print(
        results["FUSION"]
        .value_counts()
    )

    print()

In [ ]:
# Final collision + fusion summary

# Final collision + fusion summary

final_summary = []

for condition in all_data:

    spots = all_data[condition]["spots"]
    tracks = all_data[condition]["tracks"]

    # -------------------------
    # Droplet statistics
    # -------------------------

    n_droplets = spots["TRACK_ID"].nunique()

    duration = (
        spots["POSITION_T"].max()
        - spots["POSITION_T"].min()
    )

    possible_pairs = (
        n_droplets * (n_droplets - 1) / 2
    )

    # -------------------------
    # Collision statistics
    # -------------------------

    n_collisions = len(
        collision_events_unique[condition]
    )

    collision_frequency = (
        n_collisions /
        (possible_pairs * duration)
    )

    # -------------------------
    # Fusion statistics
    # -------------------------

    # TrackMate merge events
    n_merges = int(
        tracks["NUMBER_MERGES"].sum()
    )

    fusion_frequency = (
        n_merges /
        (possible_pairs * duration)
    )

    final_summary.append({

        "Condition": condition,

        "Droplets": n_droplets,

        "Duration (s)": duration,

        "Possible pairs": int(possible_pairs),

        "Collision events": n_collisions,

        "Collision frequency (events pair⁻¹ s⁻¹)":
            collision_frequency,

        "TrackMate merge events": n_merges,

        "Fusion frequency (events pair⁻¹ s⁻¹)":
            fusion_frequency

    })

final_summary_df = pd.DataFrame(final_summary)

# Optional: scaled values for easier reading
final_summary_df[
    "Collision frequency (×10⁻⁶ pair⁻¹ s⁻¹)"
] = (
    final_summary_df[
        "Collision frequency (events pair⁻¹ s⁻¹)"
    ] * 1e6
)

final_summary_df[
    "Fusion frequency (×10⁻⁶ pair⁻¹ s⁻¹)"
] = (
    final_summary_df[
        "Fusion frequency (events pair⁻¹ s⁻¹)"
    ] * 1e6
)

final_summary_df

In [ ]:
final_summary_plot = final_summary_df.copy()

final_summary_plot[
    "Collision frequency\n(×10⁻⁶ pair⁻¹ s⁻¹)"
] = (
    final_summary_plot[
        "Collision frequency\n(events pair⁻¹ s⁻¹)"
    ]
    * 1e6
)


final_summary_plot[
    "Fusion frequency\n(×10⁻⁶ droplet⁻¹ s⁻¹)"
] = (
    final_summary_plot[
        "Fusion frequency\n(events droplet⁻¹ s⁻¹)"
    ]
    * 1e6
)


final_summary_plot[
    [
        "Condition",
        "Collision frequency\n(×10⁻⁶ pair⁻¹ s⁻¹)",
        "Fusion frequency\n(×10⁻⁶ droplet⁻¹ s⁻¹)"
    ]
]